# Model Training

This notebook trains a small steering model from your labeled driving data.

Workshop story:

1. Start with the RGB camera frame.
2. Convert it into a simple lane mask using classical computer vision.
3. Train a small CNN to predict steering from that mask.

That makes it easier to explain how classical CV and learning work together.


In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from jetcar.training import PilotDataset, SmallPilotNet


/home/orin/JetCar/.venv/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
csv_path = project_root / 'data' / 'raw' / 'REPLACE_WITH_SESSION' / 'labels.csv'
image_root = csv_path.parent

dataset = PilotDataset(
    csv_path=csv_path,
    image_root=image_root,
    image_size=(96, 160),
    use_mask=True,
    crop_top_ratio=0.35,
    hsv_lower=(35, 40, 40),
    hsv_upper=(95, 255, 255),
    blur_kernel=5,
    morph_kernel=5,
)

loader = DataLoader(dataset, batch_size=16, shuffle=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SmallPilotNet(in_channels=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

print('Samples:', len(dataset), 'Device:', device)


FileNotFoundError: [Errno 2] No such file or directory: '/home/orin/JetCar/data/raw/REPLACE_WITH_SESSION/labels.csv'

In [ ]:
images, targets = next(iter(loader))
print('Batch image shape:', tuple(images.shape))
print('Batch target shape:', tuple(targets.shape))

plt.figure(figsize=(10, 4))
for i in range(min(4, len(images))):
    plt.subplot(1, 4, i + 1)
    plt.imshow(images[i, 0], cmap='gray')
    plt.title(f's={float(targets[i].item()):+.2f}')
    plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
epochs = 8
for epoch in range(epochs):
    running_loss = 0.0
    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()
        preds = model(images)
        loss = loss_fn(preds, targets)
        loss.backward()
        optimizer.step()
        running_loss += float(loss.item())

    print(f'Epoch {epoch + 1}: loss={running_loss / max(len(loader), 1):.4f}')


In [ ]:
model_path = project_root / 'models' / 'pilotnet_mask_demo.pt'
torch.save(model.state_dict(), model_path)
print('Saved model to', model_path)
